# 05 — Creating datasets, six ways

Every way this package builds a dataset, from the one that needs nothing to the
one that reads your document corpus.

| Way | Needs a model? | Grounded? |
|---|---|---|
| 1. Load a spreadsheet | no | whatever is in the file |
| 2. Adversarial attack bank | no | no |
| 3. Contexts you supply | yes | yes |
| 4. Your own documents | yes + embeddings | yes |
| 5. From a description alone | yes | no |
| 6. Growing an existing set | yes | follows the seeds |

Ways 3–6 all run the **same pipeline**: a `GoldenSource` produces goldens, an
ordered list of `Stage`s transforms them. Only the source changes.

**Ways 3–6 need a live Azure OpenAI model.** Set these first:

```
LLMINSPECTOR_AZURE_ENDPOINT
LLMINSPECTOR_API_VERSION
LLMINSPECTOR_API_KEY
LLMINSPECTOR_EMBEDDING_DEPLOYMENT     # way 4 only
```

(or pass an `azure_ad_token_provider` in code instead of the key).

## 0. The provider

One `AzureOpenAIModel` for generating, one `AzureOpenAIEmbedding` for the
document path. `critic_model` is optional — set it to a cheaper deployment and
the judging stages (filtration, context scoring) use it instead, which is where
most of a run's calls go.

In [ ]:
from llminspector.config import AzureSettings
from llminspector.models import AzureOpenAIEmbedding, AzureOpenAIModel

settings = AzureSettings.from_env()

model = AzureOpenAIModel(settings)
embedding = AzureOpenAIEmbedding(settings)

model.get_model_name(), embedding.get_model_name()

In [ ]:
from llminspector.generation import GenerationConfig

# One config drives every run below. `seed` makes a run reproducible;
# `track_usage` reports what it cost.
config = GenerationConfig(
    model=model,
    embedding=embedding,
    critic_model=None,      # falls back to `model`
    max_concurrent=5,
    seed=42,
    track_usage=True,
)

## 1. Load goldens from a spreadsheet

No model involved. The column names default to the legacy set
(`question` / `ground_truth` / `contexts`) and are overridable per call.

Every column that is **not** one of the four mapped core fields is collected
into `Golden.metadata`, and `id` round-trips — so an export you edit by hand and
reload keeps its identity and its lineage.

This workbook has `Capability` / `Sub Capability` / `Char Len` alongside the
prompt. Only `Prompt` is mapped, so the other three land in `metadata` and come
back out as columns on export — before, they vanished silently on reload.

In [ ]:
from llminspector.dataset import EvaluationDataset

seed_data = EvaluationDataset.goldens_from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx",
    input_col="Prompt",
)

print(seed_data)
print(seed_data.goldens[0].metadata)
seed_data.goldens_to_pandas().head()

## 2. Adversarial — a curated attack bank

Pure pandas: it samples and filters a bank of attack prompts by capability. No
model, no network, no stages — an attack prompt is used verbatim, because
rewriting it would change the attack.

In [ ]:
from llminspector.generation import AdversarialGenerator

adversarial = AdversarialGenerator.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx",
    capability="Jailbreak",
)

adv_result = adversarial.generate()      # sync is fine: nothing to await
print(adv_result)
adv_result.to_pandas().head()

## 3. From contexts you supply

The first way that calls a model. Hand it `list[list[str]]` — each inner list is
one context — and it writes inputs answerable from each.

`default_stages()` then runs **filter → evolve → filter → style →
expected_output**. Two orderings there are deliberate:

- evolution runs *before* a final filter pass, so a chain of rewrites that
  wanders off its context is caught rather than shipped;
- expected output runs **last**, because anything earlier can still change the
  input and a reference answer for a question that no longer exists looks like
  ground truth while scoring the wrong thing.

In [ ]:
from llminspector.generation import ContextSource, Generator, default_stages

contexts = [
    [
        "The API allows 500 requests per minute per key.",
        "Bursts up to 750 are tolerated for at most 30 seconds.",
    ],
    ["Invoices are issued on the first business day of each month."],
]

context_gen = Generator(
    ContextSource(contexts, max_goldens_per_context=2),
    default_stages(),
    config=config,
)

context_result = await context_gen.a_generate()
context_result.to_pandas()

### Tuning the chain

The three config objects are passed to `default_stages()`, not to the generator,
because they describe the *stages* rather than the run.

In [ ]:
from llminspector.generation import (
    EvolutionConfig,
    FiltrationConfig,
    StylingConfig,
)

styling = StylingConfig(
    scenario="tyre retail customer support",
    task="answer billing and delivery questions",
    input_format="a short, informal chat message",
    expected_output_format="one or two plain sentences",
)

tuned_stages = default_stages(
    # Score, repair up to twice, and drop anything still failing.
    filtration=FiltrationConfig(
        quality_threshold=0.6, max_rewrites=2, on_reject="discard"
    ),
    # Two rounds, weighted towards multi-fact reasoning.
    evolution=EvolutionConfig(
        num_evolutions=2,
        strategies={"reasoning": 3.0, "constrained": 1.0, "hypothetical": 1.0},
    ),
    styling=styling,
)

tuned = Generator(ContextSource(contexts), tuned_stages, config=config)
tuned_result = await tuned.a_generate()
tuned_result.to_pandas()

## 4. From your own documents

`DocumentSource` loads a corpus, chunks it on real tokens, embeds each document
in one call, scores candidate chunks with the critic model, and assembles
contexts from the best ones — then hands them to the **same stage chain** as
way 3.

`.txt` / `.md` / `.mdx` are read in core. PDF and DOCX need
`pip install 'llminspector[documents]'`.

In [ ]:
from llminspector.generation import ContextConfig, DocumentSource

doc_source = DocumentSource(
    directory="../tests/test_sample/corpus",
    context_config=ContextConfig(
        # Small because the shipped sample corpus is three short files.
        # A real corpus wants chunk_size=1024, chunk_overlap=128.
        chunk_size=96,
        chunk_overlap=16,
        max_contexts=5,
        chunks_per_context=2,
        similarity_threshold=0.5,   # 0.0 would accept every neighbour
        cross_file=False,
    ),
    max_goldens_per_context=2,
)

doc_result = await Generator(doc_source, default_stages(), config=config).a_generate()
doc_result.to_pandas()[["input", "expected_output", "source_file", "quality"]]

Chunk and context sizes are validated **before the first embedding call**, and
the error names your actual numbers rather than leaving you to guess:

```
The corpus is about 412 token(s), which splits into roughly 1 chunk(s) at
chunk_size=2048 — fewer than the 4 chunk(s) each context needs.
Try chunk_size=64 with chunk_overlap=12, or lower chunks_per_context.
```

Set `cross_file=True` to merge contexts drawn from *different* files, so the
generated inputs require combining documents. Chunks are then prefixed
`[SOURCE: <file>]`, but only when a context really spans more than one file.

In [ ]:
# What the run was actually grounded in, without paying for it again.
for context in doc_source.contexts:
    print(f"{context.score:.2f}  {context.source_files}  {len(context.chunks)} chunk(s)")

## 5. From a description alone

No corpus and no contexts — just a description of the setting. For evaluating a
system you have no source material for.

All three styling fields are required here, and omitting any names **every**
missing field at once rather than one per rerun. Nothing is grounded, so there
is no expected output by default: a reference answer invented without source
material is not ground truth.

In [ ]:
from llminspector.generation import ScratchSource

scratch = Generator(
    ScratchSource(styling, num_goldens=12),
    default_stages(styling=styling),
    config=GenerationConfig(
        model=model, seed=42, include_expected_output=False, track_usage=True
    ),
)

scratch_result = await scratch.a_generate()
scratch_result.to_pandas()[["input", "quality", "evolutions"]]

## 6. Growing an existing set

`SeedGoldenSource` produces more goldens in the vein of ones you already have —
the workbook from way 1, or the output of any run above.

Omit `styling` and it is **reverse-engineered** from the seeds with one call, so
the augmented set sounds like the set it grew from.

Seeds are **partitioned**: those carrying context go down the grounded path,
those without go down the scratch path, and both run. Routing the whole batch on
whether *any* seed has context — the obvious implementation — silently drops
every context-free seed.

In [ ]:
from llminspector.generation import SeedGoldenSource

grown = Generator(
    SeedGoldenSource(seed_data.goldens[:5], max_per_golden=2),
    default_stages(),
    config=config,
)

grown_result = await grown.a_generate()
grown_result.to_pandas()[["input", "generated_from", "seed_id"]].head()

## 7. Reading the result

`GenerationResult` is shaped after `EvaluationResult`, so there is one
partial-failure idiom to learn. A run always returns; the reasons live on it.

**`rejected` and `errors` are different things.** A golden that did not clear
the quality bar is *rejected*; a stage that blew up is an *error*. They need
different responses from whoever reads the run.

In [ ]:
print(context_result)                 # produced / rejected / failed
print(context_result.error_summary())

context_result.rejected[:3]           # {'index', 'stage', 'reason'}

In [ ]:
# Lineage: what happened to one golden, stage by stage.
golden = context_result.goldens[0]
for entry in golden.metadata["lineage"]:
    print(entry)

In [ ]:
# What the run cost (track_usage=True). Reasks are counted.
context_result.usage

The output column set is knowable **before** a run — which, for an LLM-backed
pipeline, means without paying for one.

In [ ]:
context_gen.metadata_keys

### Optional: roughen the inputs

`PerturbationStage` makes no model call. It belongs *after* everything else —
filtration would score its own noise as a defect.

In [ ]:
from llminspector.generation import PerturbationStage

noisy = Generator(
    ContextSource(contexts),
    [*default_stages(), PerturbationStage(["typo", "ocr_typo"])],
    config=config,
)
noisy_result = await noisy.a_generate()
noisy_result.to_pandas()[["input", "perturbation"]]

## 8. Export, answer, evaluate

The goldens are seeds: run them through *your* application to get answers, then
score. `to_test_cases()` is the in-memory path; `goldens_to_excel` is the
round-trip path, and metadata survives it.

In [ ]:
from llminspector.dataset import EvaluationDataset

generated = EvaluationDataset(goldens=doc_result.goldens)
generated.goldens_to_excel("generated_goldens.xlsx")

# ...run your application over generated.goldens...
answers = ["(your system's answer)" for _ in generated.goldens]

scored_dataset = EvaluationDataset(test_cases=generated.to_test_cases(answers=answers))
scored_dataset.test_cases[0].golden_id == generated.goldens[0].id

In [ ]:
from llminspector import a_evaluate
from llminspector.metrics import AnswerCorrectnessMetric, FaithfulnessMetric

result = await a_evaluate(
    scored_dataset,
    metrics=[FaithfulnessMetric(model), AnswerCorrectnessMetric(model)],
)
result.to_excel("generated_scored.xlsx")
result.to_pandas().head()

`golden_id` carries through, so every scored row traces back to the golden — and
to the lineage — that produced it.

Next: [04 — end to end](04_end_to_end.ipynb).